# User study: Sim2Real Counterfactual Comparison

Same four phases as the CoXAM tutorial, but **Phase 2 is skipped entirely**:
Sim2Real simulates from a fixed published corpus in `assets/` -- explanation
attributions, AI predictions and ground-truth answer keys are already there.
No dataset is prepared and no AI model is trained; `run_sim2real_study` reads
neither `study.data` nor `study.trained_ai_model`.

Phase 5 adds a comparison this tutorial's CoXAM counterpart does not have: the
real study participants beside the model fitted to them, from
`assets/human_data/Sim2Real/`.

## Phase 0: Set up your environment

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import src as xk

OUTPUT_DIR = REPO_ROOT / "tutorials" / "experiment_output" / "sim2real_workflow_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Phase 1: Configure the User Study

### Step 1.1 Initialize the study workflow

In [ ]:
xaikitTest = xk.xaikitTest("sim2real_counterfactual_study", output_dir=OUTPUT_DIR)

### Step 1.2 Define the user study variables

`xai_property` is Sim2Real's own IV -- faithful / sparse / robust /
sparse_robust -- and is the discriminator the server uses to tell a Sim2Real
design apart from a plain CoAX one (see `server/README.md`). The DV is
`counterfactual_accuracy`: whether the model's "higher or lower" answer
matches the published answer key.

In [ ]:
XAI_PROPERTIES = ["faithful", "sparse", "robust", "sparse_robust"]

xaikitTest.add_iv("xai_property", "within", XAI_PROPERTIES, randomization="block")
xaikitTest.add_cv("user_task", ["counterfactual_simulation"])
xaikitTest.add_dv("counterfactual_accuracy", ["continuous"])

xaikitTest.validate_design(show=True)

## Phase 2: Prepare AI Model and Explanations

**Skipped.** Sim2Real's stimuli, attributions and ground-truth labels are a
fixed corpus (`Sim2RealAttributionProjector`), not something trained or
generated here. `run_sim2real_study`'s own docstring is explicit that it reads
neither `study.data` nor `study.trained_ai_model`.

## Phase 3: Run the User Study Simulation

### Step 3.1 Build trials directly from the corpus

The generic `study.generate_trials()` requires a prepared dataset regardless
of participant runner, so trials are built directly here instead: every
participant sees the same 29 published test cases, one participant per
`xai_property` condition.

In [ ]:
from src.virtual_experiment_executor.experiment_simualtion.Sim2Real.sim2real_trial_executor import (
    sim2real_available_instance_ids,
)

N_PARTICIPANTS_PER_CONDITION = 3
test_instance_ids = sim2real_available_instance_ids(split="test")
print(f"published test instances: {len(test_instance_ids)}")

rows = []
participant_id = 1
for xai_property in XAI_PROPERTIES:
    for _ in range(N_PARTICIPANTS_PER_CONDITION):
        for instance_id in test_instance_ids:
            rows.append({
                "participantId": participant_id,
                "instanceId": instance_id,
                "xai_property": xai_property,
                "phase": "testing",
            })
        participant_id += 1

xaikitTest.trials = pd.DataFrame(rows)
print(f"trials: {len(xaikitTest.trials)} rows, "
      f"{xaikitTest.trials['participantId'].nunique()} participants")

### Step 3.2 Run the virtual study simulation

`strategy="attribution_sum"` is the population-best fit wherever the changed
feature has a visible attribution; `cognitive_params` overrides its defaults
(reproduces the values a UI export's `cognitiveConfig` would set).

In [ ]:
from src.virtual_experiment_executor.experiment_simualtion.Sim2Real.sim2real_study_runner import (
    run_sim2real_study,
)

simulated_results = run_sim2real_study(
    xaikitTest,
    mode="whole_experiment",
    strategy="attribution_sum",
    cognitive_params={
        "max_features_attended": 4,
        "aggregation": "value_weighted",
        "confidence_intercept": -1.5,
    },
    normalize_by_i_max=True,
)

print("simulated rows:", len(simulated_results))
simulated_results[[
    "participantId", "instanceId", "xai_property", "agent_response_increases",
    "ground_truth_increases", "counterfactual_accuracy", "confidence_delta",
]].head(8)

Save the study simulation results

In [ ]:
csv_path, json_path = xaikitTest.save_results(out_dir="simulated_results")
print("saved:", csv_path)

## Phase 4: Analysis of Simulation Results

### Step 4.1 Compute descriptive statistics and plot results grid

In [ ]:
# counterfactual_accuracy is already filled in by the executor -- any DV whose
# name mentions "accuracy" is scored against the corpus's ground-truth label.
xaikitTest.simulated_results = simulated_results

analysis = xaikitTest.analyze_iv_dv(iv="xai_property", dv="counterfactual_accuracy")
analysis

In [ ]:
xaikitTest.plot_results_grid(
    ivs=["xai_property"],
    dvs=["counterfactual_accuracy"],
    phase=None,
    title="Sim2Real simulated counterfactual accuracy",
)

## Phase 5: Human vs Sim2Real, on the Real Study Participants

Sections 1-4 simulate a *new* run. This section is the study's own fitted
result: 46 real participants beside the Sim2Real model fitted to them, from
`assets/human_data/Sim2Real/` -- built the same way as the CoXAM and CoAX
comparisons (`src/result_visualizer/study_comparisons.py`).

In [ ]:
from src.result_visualizer import human_vs_model_report, study_comparison

report_path = human_vs_model_report("sim2real", OUTPUT_DIR / "human_vs_sim2real.html")
print("wrote", report_path)

sim2real = study_comparison("sim2real")
print(sim2real.name, "|", sim2real.task)
print("participants:", sim2real.participants, "| panels:", len(sim2real.panels))
for panel in sim2real.panels:
    print(f"\n{panel.title}")
    print(panel.to_frame().to_string(index=False))

## What This Notebook Shows

- Sim2Real runs end to end with **no dataset preparation and no AI training** --
  `run_sim2real_study` reads only the fixed published corpus and the trials
  built directly from it.
- Trials are built by hand rather than through `generate_trials()`, because
  that generic call needs a prepared dataset regardless of participant runner
  -- a gap documented in `server/README.md`, not specific to notebooks.
- `strategy` and `cognitive_params` are the same UI-facing controls the server
  accepts under `sim2real_params`, resolved through
  `normalize_cognitive_params` (see `design_export.py`).
- Phase 5's comparison reads the real 46-participant fit, not a fresh
  simulation -- it is what the paper reports, and is a different question from
  "does today's simulation run," which Phase 3-4 answer.